# YOLO v11 — Live Video Detection

```

Model trained on: https://colab.research.google.com/drive/1VRtxhdpRhymDpmeAQz_132AKRgMhve5_?usp=sharing

In [1]:

import os
import glob
import time
import cv2
import numpy as np
from ultralytics import YOLO

In [2]:


MODEL_PATH = "best.torchscript"

SOURCE = "/Users/davidcervantes/Documents/Sistemas Multiagentes/Proyecto Gym/Model/Grabación de pantalla 2026-04-19 a la(s) 9.11.15 p.m..mov"  # ← Change to a video filename to use a file instead

CONFIDENCE_THRESHOLD = 0.25
WINDOW_NAME = "YOLO v11 — Live Detection (press Q to quit)"

In [3]:


if isinstance(SOURCE, str) and SOURCE == "auto":
    VIDEO_EXTENSIONS = ("*.mp4", "*.avi", "*.mov", "*.mkv", "*.webm")
    video_files = []
    for ext in VIDEO_EXTENSIONS:
        video_files.extend(glob.glob(ext))
    if not video_files:
        raise FileNotFoundError(
            "No video file found. Place a video in the Model/ folder "
            "or set SOURCE = 0 to use your webcam."
        )
    SOURCE = video_files[0]
    print(f"Auto-detected video: {SOURCE}")

if isinstance(SOURCE, int):
    print(f"Source: Webcam (device {SOURCE})")
else:
    print(f"Source: Video file — {SOURCE}")

Source: Video file — /Users/davidcervantes/Documents/Sistemas Multiagentes/Proyecto Gym/Model/Grabación de pantalla 2026-04-19 a la(s) 9.11.15 p.m..mov


In [4]:

model = YOLO(MODEL_PATH)
print("Model loaded successfully!")
print(f"   Class names: {model.names}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Model loaded successfully!
Loading best.torchscript for TorchScript inference...
   Class names: {0: 'machine-unused', 1: 'machine-used'}


In [ ]:


# Color palette for bounding boxes
np.random.seed(42)
COLORS = np.random.randint(50, 255, size=(80, 3), dtype=np.uint8)

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError(f"Could not open source: {SOURCE}")

# Get video properties
src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"🎬  Stream: {width}x{height} @ {src_fps:.0f} FPS")

frame_count = 0
fps_display = 0.0
prev_time = time.time()

print("\nLive window opened — press 'Q' or 'ESC' to close.\n")

while True:
    ret, frame = cap.read()

    # If video file ended, loop back to start
    if not ret:
        if isinstance(SOURCE, str):
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            continue
        else:
            break

    frame_count += 1

    # ── Run YOLO inference ──
    results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)

    # ── Draw detections ──
    det_count = 0
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf  = float(box.conf[0])
            cls_id = int(box.cls[0])
            cls_name = model.names.get(cls_id, str(cls_id))
            color = tuple(int(c) for c in COLORS[cls_id % len(COLORS)])

            # Bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Label with background
            label = f"{cls_name} {conf:.2f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 6, y1), color, -1)
            cv2.putText(frame, label, (x1 + 3, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
            det_count += 1

    # ── FPS counter ──
    curr_time = time.time()
    fps_display = 1.0 / max(curr_time - prev_time, 1e-6)
    prev_time = curr_time

    # ── HUD overlay ──
    hud = f"FPS: {fps_display:.1f}  |  Detections: {det_count}"
    cv2.rectangle(frame, (0, 0), (len(hud) * 13 + 10, 36), (0, 0, 0), -1)
    cv2.putText(frame, hud, (8, 26),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)

    # ── Show frame ──
    cv2.imshow(WINDOW_NAME, frame)

    # ── Check for quit ──
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:  # Q or ESC
        break

cap.release()
cv2.destroyAllWindows()
print(f"\nStopped. Processed {frame_count} frames.")

🎬  Stream: 2940x1642 @ 30 FPS

Live window opened — press 'Q' or 'ESC' to close.

Loading best.torchscript for TorchScript inference...

Stopped. Processed 96 frames.


: 